# Vapour Compression HPR Targets and Performance Maps

**Learning outcome:** Apply vapour compression hpr targets and performance maps through the public `PinchProblem` or `PinchWorkspace` workflow.

**Level:** Advanced  
**Execution profile:** `slow-hpr`  
**Expected runtime:** 5 to 30 minutes  
**Optional extras:** hpr

## Study question and data

**Study question:** How does a successful HPR target become a versioned part-load map for downstream utility optimization?

## Step 1: Build a default CoolProp target

This cell defines `screen_optional_hpr()`, which returns an `hpr_summary()` or a failure record instead of raising. It then runs `vapour_compression_heat_pump()` with water as refrigerant. `simulation_backend` is omitted, so CoolProp is used. `coolprop_summary` holds the headline results of this target.

In [ ]:
from OpenPinch import PinchProblem
from OpenPinch.contracts.hpr import HPRTargetingError
from OpenPinch.contracts.hpr_performance_map import (
    HprPerformanceMapRequest,
)
from OpenPinch.presentation.reporting.hpr import hpr_failure_summary, hpr_summary


def screen_optional_hpr(method, **arguments):
    try:
        result = method(**arguments)
        if result is None:
            return {"status": "method unavailable", "reason": "no target returned"}
        return hpr_summary(result, method.__name__)
    except HPRTargetingError as error:
        return hpr_failure_summary(error)
    except ImportError as error:
        return {"status": "optional dependency unavailable", "reason": str(error)}
    except NotImplementedError as error:
        return {"status": "method unavailable", "reason": str(error)}

problem = PinchProblem("basic_pinch.json", project_name="HPR Models")
working_fluid_examples = {
    "pure": "water",
    "registered_blend": "R410A.mix",
}
# Omitting simulation_backend selects the CoolProp default.
coolprop_target = problem.target.vapour_compression_heat_pump(
    refrigerants=[working_fluid_examples["pure"]],
    load_fraction=0.25,
    condensers=1,
    evaporators=1,
    maximum_restarts=1,
    maximum_iterations=20,
    maximum_evaluations=50,
)
assert coolprop_target.hpr_success
coolprop_summary = hpr_summary(coolprop_target, "default CoolProp heat pump")

## Step 2: Generate and export a target-owned part-load map

This cell reads the target's `target_simulation_record` and derives one source and one sink temperature from it. An `HprPerformanceMapRequest` asks for load fractions of 0.50, 0.75 and 1.00. `hpr_performance_map()` generates the map from `coolprop_target`. `plain_performance_map` and `target_record_json` hold JSON-ready exports of the map and record.

In [ ]:
target_simulation_record = (
    coolprop_target.hpr_details.target_simulation_record
)
assert target_simulation_record is not None
assert target_simulation_record.simulation_backend == "coolprop"
assert target_simulation_record.nominal_useful_duty > 0.0
source_temperature = (
    target_simulation_record.nominal_evaporating_temperature
    + target_simulation_record.source_approach_temperature
)
sink_temperature = (
    target_simulation_record.nominal_condensing_temperature
    - target_simulation_record.sink_approach_temperature
)
map_request = HprPerformanceMapRequest(
    map_id="notebook-09-coolprop-map",
    source_temperatures=[source_temperature],
    sink_temperatures=[sink_temperature],
    load_fractions=[0.50, 0.75, 1.00],
)
performance_map = problem.target.hpr_performance_map(
    target=coolprop_target, request=map_request
)
plain_performance_map = performance_map.model_dump(mode="json")
target_record_json = target_simulation_record.model_dump(mode="json")
assert plain_performance_map["points"]

## Step 3: Screen explicit TESPy and molar-mixture intent

This cell requests `vapour_compression_heat_pump()` with `simulation_backend="tespy"` and an explicit 50/50 R32/R125 molar mixture. `initialize_from_carnot` and `allow_integrated_expander` are both `False`. The call runs through `screen_optional_hpr()`, so `tespy_target` holds either a summary or a failure record. A TESPy failure does not fall back to CoolProp.

In [ ]:
# Explicit N-component mixtures use component[mole_fraction].
explicit_molar_mixture = "HEOS::R32[0.5]&R125[0.5]"
tespy_target = screen_optional_hpr(
    problem.target.vapour_compression_heat_pump,
    simulation_backend="tespy",
    refrigerants=[explicit_molar_mixture],
    initialize_from_carnot=False,
    allow_integrated_expander=False,
    load_fraction=0.25,
    condensers=1,
    evaporators=1,
    maximum_restarts=1,
    maximum_iterations=20,
    maximum_evaluations=50,
)
# TESPy is a separate selection: failure never falls back to CoolProp.
tespy_no_fallback = tespy_target
tespy_no_fallback

## Step 4: Compare refrigeration and Brayton variants

This cell runs `vapour_compression_refrigeration()` with ammonia, using the same load fraction, stage counts and search limits as earlier. `vc_refrigeration_summary` holds its headline results. `brayton` and `brayton_refrigeration` screen the Brayton heat pump and refrigeration methods through `screen_optional_hpr()`. Each holds a summary or a failure record.

In [ ]:
vc_refrigeration = problem.target.vapour_compression_refrigeration(
    refrigerants=["ammonia"],
    load_fraction=0.25,
    condensers=1,
    evaporators=1,
    maximum_restarts=1,
    maximum_iterations=20,
    maximum_evaluations=50,
)
assert vc_refrigeration.hpr_success
vc_refrigeration_summary = hpr_summary(
    vc_refrigeration, "vapour compression refrigeration"
)
brayton = screen_optional_hpr(
    problem.target.brayton_heat_pump,
    load_fraction=0.25, maximum_restarts=1,
    maximum_iterations=20, maximum_evaluations=50,
)
brayton_refrigeration = screen_optional_hpr(
    problem.target.brayton_refrigeration,
    load_fraction=0.25, maximum_restarts=1,
    maximum_iterations=20, maximum_evaluations=50,
)

## Review the result

Inspect the detached winning record and the complete plain-data map before comparing optional TESPy, refrigeration, or Brayton screens. A failed optional screen is evidence, not permission to relabel a CoolProp result as TESPy.

In [ ]:
from IPython.display import display

map_preview = {
        "schema_version": plain_performance_map["schema_version"],
        "thermodynamic_backend": plain_performance_map[
            "thermodynamic_backend"
        ],
        "reference_capacity_basis": plain_performance_map[
            "reference_capacity_basis"
        ],
        "points": plain_performance_map["points"],
}
cycle_comparison = {
    "default CoolProp heat pump": coolprop_summary,
    "explicit TESPy mixture heat pump": tespy_target,
    "vapour compression refrigeration": vc_refrigeration_summary,
    "Brayton heat pump": brayton,
    "Brayton refrigeration": brayton_refrigeration,
}
display(coolprop_summary)
display(map_preview)
display(cycle_comparison)

## Interpret the result

Treat the winning simulation record as the immutable bridge from targeting to map generation. CoolProp remains the default; selecting TESPy is explicit and never implies fallback. Pure fluids, provider-registered blends, and explicit molar mixtures still depend on installed property support and feasible dew/bubble states.

## Adapt this template

Replace the sample with a reviewed HPR target, keep the one-evaporator/one-condenser schema 1.0 boundary, choose representative temperature and load grids, and pass only the plain versioned export to OpenUtility or another optimizer.